# 08.7 Prompt 控制与生成结果评估

本 Notebook 读取当前已经生成的音频和 runner 状态表，演示如何记录 prompt、计算基础无参考指标、导出 MOS 条目，并检查 CTIS/ChMusic/audio_author 等中文案例资产。


In [ ]:
from pathlib import Path
import os
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from _common.audio_io import load_audio
from _common.dataset_registry import asset_status
from _common.device_utils import choose_device
from _common.paths import portable_path
from _common.plotting import finish_figure, setup_plot_style
from _common.tables import write_rows
from evaluation.audio_metrics import (
    clipping_ratio,
    fad_proxy,
    loudness_lufs,
    peak_amplitude,
    rms,
    silence_ratio,
    spectral_centroid,
    spectral_flatness,
    zero_crossing_rate,
)
from evaluation.clap_score import (
    check_clap_dependency,
    compute_clap_scores,
    write_clap_scores,
    write_clap_status,
)
from evaluation.comparison_table import build_model_comparison_rows, write_model_comparison
from evaluation.mos_export import write_mos_items

OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_METRICS = ROOT / "outputs" / "metrics"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
for path in [OUTPUT_FIGURES, OUTPUT_METRICS, OUTPUT_TABLES]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()
DEVICE = choose_device(os.getenv("CHAPTER08_DEVICE", "auto"))
print("selected device:", DEVICE)

def rel(path):
    return portable_path(path, ROOT)


In [ ]:
prompt_grid = pd.DataFrame(
    [
        {"prompt_id": "cn_plucked_sparse", "prompt_cn": "稀疏的中国弹拨乐", "prompt_en": "sparse Chinese plucked strings", "control_axis": "instrumentation"},
        {"prompt_id": "piano_minimal", "prompt_cn": "极简钢琴独奏", "prompt_en": "minimal solo piano", "control_axis": "texture"},
        {"prompt_id": "orchestra_slow", "prompt_cn": "缓慢的弦乐队", "prompt_en": "slow string orchestra", "control_axis": "ensemble"},
    ]
)
display(prompt_grid)
write_rows(OUTPUT_TABLES / "08_7_prompt_grid.csv", prompt_grid.to_dict("records"))


In [ ]:
asset_rows = []
for asset_id in ["ctis", "chmusic", "audio_author_ch08"]:
    status = asset_status(asset_id)
    asset_rows.append(
        {
            "asset_id": asset_id,
            "available": status.ok,
            "paths": ";".join(rel(path) for path in status.existing_paths),
            "download_hint": status.spec.download_hint,
        }
    )
display(pd.DataFrame(asset_rows))
write_rows(OUTPUT_TABLES / "08_7_case_asset_status.csv", asset_rows)


In [ ]:
audio_files = sorted((ROOT / "output_audio").glob("08_*/*.wav"))
print("audio files for metric demo:", len(audio_files))
for path in audio_files[:10]:
    print("-", rel(path))


In [ ]:
metric_rows = []
loaded_audio = []
for path in audio_files:
    audio, sr = load_audio(path, mono=True)
    loaded_audio.append((audio, sr, path))
    metric_rows.append(
        {
            "audio_path": rel(path),
            "sample_rate": sr,
            "duration_sec": len(audio) / sr,
            "peak_amplitude": peak_amplitude(audio),
            "rms": rms(audio),
            "silence_ratio": silence_ratio(audio),
            "clipping_ratio": clipping_ratio(audio),
            "loudness_lufs": loudness_lufs(audio),
            "zero_crossing_rate": zero_crossing_rate(audio),
            "spectral_flatness": spectral_flatness(audio),
            "spectral_centroid": spectral_centroid(audio, sr),
        }
    )
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)
write_rows(OUTPUT_METRICS / "08_7_audio_metrics.csv", metric_rows)


In [ ]:
if not metrics_df.empty:
    metric_panels = [
        ("peak_amplitude", "峰值幅度"),
        ("rms", "RMS 电平"),
        ("silence_ratio", "静音比例"),
        ("clipping_ratio", "削波比例"),
        ("spectral_flatness", "频谱平坦度"),
        ("zero_crossing_rate", "过零率"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for ax, (column, title) in zip(axes.flat, metric_panels):
        plot_df = metrics_df.sort_values(column)
        ax.barh(plot_df["audio_path"].map(lambda x: Path(x).name), plot_df[column], color="0.35")
        ax.set_title(title)
        ax.set_xlabel(title)
        ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=3))
        ax.tick_params(axis="both", labelsize=8)
    finish_figure(fig, OUTPUT_FIGURES / "08_7_audio_metric_flatness.png")
    plt.show()


In [ ]:
if len(loaded_audio) >= 2:
    midpoint = max(1, len(loaded_audio) // 2)
    ref = [item[0] for item in loaded_audio[:midpoint]]
    gen = [item[0] for item in loaded_audio[midpoint:]]
    proxy = fad_proxy(ref, gen, loaded_audio[0][1])
    fad_rows = [{"reference_count": len(ref), "generated_count": len(gen), "fad_proxy": proxy}]
    display(pd.DataFrame(fad_rows))
    write_rows(OUTPUT_METRICS / "08_7_fad_proxy.csv", fad_rows)
else:
    print("Need at least two audio files for FAD proxy demo.")


In [ ]:
clap_status = check_clap_dependency()
write_clap_status(OUTPUT_METRICS / "08_7_clap_status.csv", clap_status)
display(pd.DataFrame([clap_status.as_row()]))

RUN_CLAP = os.getenv("CHAPTER08_RUN_CLAP", "0") == "1"
if clap_status.available and RUN_CLAP and audio_files:
    prompt_rows = [
        {"prompt_id": row["prompt_id"], "prompt": row["prompt_en"]}
        for row in prompt_grid.to_dict("records")
    ]
    scores = compute_clap_scores(
        audio_files[:8],
        prompt_rows,
        device=DEVICE,
    )
    write_clap_scores(OUTPUT_METRICS / "08_7_clap_scores.csv", scores)
    display(pd.DataFrame([score.as_row() for score in scores]).head())
else:
    print(clap_status.reason)
    print(clap_status.next_action)
    print("Set CHAPTER08_RUN_CLAP=1 after CLAP dependencies and checkpoints are ready.")


In [ ]:
mos_rows = [
    {
        "item_id": f"item_{idx:03d}",
        "model_name": Path(row["audio_path"]).parent.name,
        "prompt_id": "",
        "audio_path": row["audio_path"],
        "question": "Rate audio quality and prompt fit from 1 to 5.",
        "notes": "",
    }
    for idx, row in enumerate(metric_rows, start=1)
]
write_mos_items(OUTPUT_TABLES / "08_7_mos_items.csv", mos_rows)
display(pd.DataFrame(mos_rows).head())


In [ ]:
runner_status_path = OUTPUT_TABLES / "08_model_runner_status.csv"
runner_rows = pd.read_csv(runner_status_path).to_dict("records") if runner_status_path.exists() else []

fad_path = OUTPUT_METRICS / "08_7_fad_proxy.csv"
fad_value = ""
if fad_path.exists():
    fad_df = pd.read_csv(fad_path)
    if not fad_df.empty:
        fad_value = fad_df.iloc[0].get("fad_proxy", "")

clap_scores_path = OUTPUT_METRICS / "08_7_clap_scores.csv"
clap_rows = pd.read_csv(clap_scores_path).to_dict("records") if clap_scores_path.exists() else []

comparison_rows = build_model_comparison_rows(
    metric_rows=metric_rows,
    runner_status_rows=runner_rows,
    fad_proxy_value=fad_value,
    clap_score_rows=clap_rows,
)
write_model_comparison(OUTPUT_TABLES / "08_model_comparison.csv", comparison_rows)
display(pd.DataFrame(comparison_rows))
print("Wrote model comparison rows:", len(comparison_rows))
